# ACT's complete flow:
Current_State + Action_Sequence → Encoder → CLS_Output → Latent_Z → Decoder → Final_Actions

In [ ]:
'''
# Training: Learn action distributions given current state + demonstrated actions
encoder_input = [CLS, current_qpos, action_sequence]
# Model learns: "Given where I am now + what I should do, refine the actions"

# During execution:
encoder_input = [CLS, current_qpos, zeros/previous_actions]  
# Model can adapt based on CURRENT state, not just past history

# Example: Robot is reaching for cup, suddenly cup moves

# ACT Approach:
current_qpos = [new_arm_position]  # Current state after cup moved
# Model immediately gets updated state information
# Can adapt action sequence based on NEW current position

# Old Approach:  
state_history = [old_positions × 99, new_position × 1]
# Still heavily influenced by 99 old positions
# Takes longer to adapt to sudden environmental changes
'''

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import time
import math
import matplotlib.pyplot as plt
import torch.nn.functional as F
from torchvision import transforms, models
from PIL import Image
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
import os
import h5py
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

def load_hdf5(dataset_path):
    if not os.path.isfile(dataset_path):
        print(f'Dataset does not exist at \n{dataset_path}\n')
        return None, None, None, None

    with h5py.File(dataset_path, 'r') as root:
        qpos = root['/observations/qpos'][()]
        qvel = root['/observations/qvel'][()]
        image_dict = dict()
        for cam_name in root[f'/observations/images/'].keys():
            image_dict[cam_name] = root[f'/observations/images/{cam_name}'][()]
        action = root['/action'][()]

    return qpos, qvel, action, image_dict

def load_all_episodes(data_dir):
    """
    Load all HDF5 files from the specified directory
    Args:
        data_dir: directory containing HDF5 files
    Returns:
        combined qpos, qvel, action arrays and list of image dicts
    """
    print(f"Loading episodes from: {data_dir}")
    
    # Find all HDF5 files
    hdf5_files = [f for f in os.listdir(data_dir) if f.endswith('.hdf5')]
    hdf5_files.sort()  # Sort for consistent ordering
    
    if not hdf5_files:
        print("No HDF5 files found in the directory!")
        return None, None, None, None
    
    print(f"Found {len(hdf5_files)} HDF5 files: {hdf5_files}")
    
    all_qpos = []
    all_qvel = []
    all_action = []
    all_image_dicts = []
    
    episode_info = []
    
    for filename in hdf5_files:
        filepath = os.path.join(data_dir, filename)
        print(f"Loading {filename}...")
        
        qpos, qvel, action, image_dict = load_hdf5(filepath)
        
        if qpos is not None:
            print(f"  - Episode length: {len(qpos)} timesteps")
            print(f"  - QPos shape: {qpos.shape}")
            print(f"  - Action shape: {action.shape}")
            
            all_qpos.append(qpos)
            all_qvel.append(qvel)
            all_action.append(action)
            all_image_dicts.append(image_dict)
            
            episode_info.append({
                'filename': filename,
                'length': len(qpos),
                'start_idx': sum(len(ep) for ep in all_qpos[:-1]),
                'end_idx': sum(len(ep) for ep in all_qpos)
            })
        else:
            print(f"  - Failed to load {filename}")
    
    if not all_qpos:
        print("No valid episodes loaded!")
        return None, None, None, None
    
    # Concatenate all episodes
    combined_qpos = np.concatenate(all_qpos, axis=0)
    combined_qvel = np.concatenate(all_qvel, axis=0)
    combined_action = np.concatenate(all_action, axis=0)
    
    print(f"\nCombined dataset:")
    print(f"Total episodes: {len(all_qpos)}")
    print(f"Total timesteps: {len(combined_qpos)}")
    print(f"Combined QPos shape: {combined_qpos.shape}")
    print(f"Combined Action shape: {combined_action.shape}")
    
    # Print episode breakdown
    print(f"\nEpisode breakdown:")
    for info in episode_info:
        print(f"  {info['filename']}: {info['length']} steps (indices {info['start_idx']}-{info['end_idx']-1})")
    
    return combined_qpos, combined_qvel, combined_action, all_image_dicts

# Load all episodes from the directory
data_dir = r'C:\Users\Administrator\Documents\transformer\ACT-Shaka\data\task1c'
qpos, qvel, action, image_dict_list = load_all_episodes(data_dir)

if qpos is None:
    print("Failed to load any data!")
    exit()

# Create DataFrames with joint headers
joint_headers = ['timestamp', 'b', 's', 'e', 't', 'r', 'g']

# Create timestamp column (assuming sequential timesteps across all episodes)
num_timesteps = len(qpos)
timestamps = np.arange(num_timesteps)

# Create qpos DataFrame
qpos_data = np.column_stack([timestamps, qpos])
qpos_df = pd.DataFrame(qpos_data, columns=joint_headers)

# Create action DataFrame
action_data = np.column_stack([timestamps, action])
action_df = pd.DataFrame(action_data, columns=joint_headers)

print("\nDataFrame Summary:")
print("QPos DataFrame shape:", qpos_df.shape)
print("Action DataFrame shape:", action_df.shape)
print("\nQPos DataFrame head:")
print(qpos_df.head())
print("\nAction DataFrame head:")
print(action_df.head())

# Check the data ranges for each joint
print("\nQPos data ranges:")
for col in joint_headers[1:]:  # Skip timestamp
    print(f"{col}: {qpos_df[col].min():.3f} to {qpos_df[col].max():.3f}")

print("\nAction data ranges:")
for col in joint_headers[1:]:  # Skip timestamp
    print(f"{col}: {action_df[col].min():.3f} to {action_df[col].max():.3f}")

# Optional: Plot data distribution across episodes
plt.figure(figsize=(15, 10))

joint_cols = ['b', 's', 'e', 't', 'r', 'g']
joint_names = ['Base', 'Shoulder', 'Elbow', 'Wrist1', 'Wrist2', 'Gripper']

for i, (joint, joint_name) in enumerate(zip(joint_cols, joint_names)):
    plt.subplot(2, 3, i+1)
    plt.plot(qpos_df['timestamp'], qpos_df[joint], 'b-', alpha=0.7, label=f'QPos {joint}')
    plt.plot(action_df['timestamp'], action_df[joint], 'r-', alpha=0.7, label=f'Action {joint}')
    plt.title(f'{joint_name} Joint ({joint}) - All Episodes')
    plt.xlabel('Timestep')
    plt.ylabel('Joint Value (radians)')
    plt.legend()
    plt.grid(True, alpha=0.3)

plt.suptitle('Joint Data Across All Episodes', fontsize=16)
plt.tight_layout()
plt.show()

print("\nData loading complete! Now you can proceed with preprocessing and training.")

## Backbone: Resnet 34

In [ ]:
class PositionEmbeddingSine(nn.Module):
    """ACT's sinusoidal position embedding for images"""
    def __init__(self, num_pos_feats=64, temperature=10000, normalize=False, scale=None):
        super().__init__()
        self.num_pos_feats = num_pos_feats
        self.temperature = temperature
        self.normalize = normalize
        if scale is not None and normalize is False:
            raise ValueError("normalize should be True if scale is passed")
        if scale is None:
            scale = 2 * math.pi
        self.scale = scale

    def forward(self, tensor):
        """
        Args:
            tensor: [batch, channels, height, width]
        Returns:
            pos: [batch, num_pos_feats*2, height, width]
        """
        x = tensor
        # Create a mask of ones (since we don't have actual masks)
        not_mask = torch.ones_like(x[0, [0]])  # [1, height, width]
        
        y_embed = not_mask.cumsum(1, dtype=torch.float32)
        x_embed = not_mask.cumsum(2, dtype=torch.float32)
        
        if self.normalize:
            eps = 1e-6
            y_embed = y_embed / (y_embed[:, -1:, :] + eps) * self.scale
            x_embed = x_embed / (x_embed[:, :, -1:] + eps) * self.scale

        dim_t = torch.arange(self.num_pos_feats, dtype=torch.float32, device=x.device)
        dim_t = self.temperature ** (2 * (dim_t // 2) / self.num_pos_feats)

        pos_x = x_embed[:, :, :, None] / dim_t
        pos_y = y_embed[:, :, :, None] / dim_t
        pos_x = torch.stack((pos_x[:, :, :, 0::2].sin(), pos_x[:, :, :, 1::2].cos()), dim=4).flatten(3)
        pos_y = torch.stack((pos_y[:, :, :, 0::2].sin(), pos_y[:, :, :, 1::2].cos()), dim=4).flatten(3)
        pos = torch.cat((pos_y, pos_x), dim=3).permute(0, 3, 1, 2)
        return pos
    
class PositionEmbeddingLearned(nn.Module):
    """ACT's learnable position embedding for images"""
    def __init__(self, num_pos_feats=256):
        super().__init__()
        self.row_embed = nn.Embedding(50, num_pos_feats)
        self.col_embed = nn.Embedding(50, num_pos_feats)
        self.reset_parameters()

    def reset_parameters(self):
        nn.init.uniform_(self.row_embed.weight)
        nn.init.uniform_(self.col_embed.weight)

    def forward(self, tensor):
        """
        Args:
            tensor: [batch, channels, height, width]
        Returns:
            pos: [batch, num_pos_feats*2, height, width]
        """
        x = tensor
        h, w = x.shape[-2:]
        i = torch.arange(w, device=x.device)
        j = torch.arange(h, device=x.device)
        x_emb = self.col_embed(i)
        y_emb = self.row_embed(j)
        pos = torch.cat([
            x_emb.unsqueeze(0).repeat(h, 1, 1),
            y_emb.unsqueeze(1).repeat(1, w, 1),
        ], dim=-1).permute(2, 0, 1).unsqueeze(0).repeat(x.shape[0], 1, 1, 1)
        return pos
    

In [ ]:
# ========================================
# OFFICIAL ACT BACKBONE IMPLEMENTATION
# Exact replica of the official ACT repository workflow
# ========================================

import torch
import torch.nn as nn
import torchvision
import math
import numpy as np
from collections import OrderedDict
from torchvision.models._utils import IntermediateLayerGetter

# ========================================
# 1. FROZEN BATCH NORM (Official ACT)
# ========================================

class FrozenBatchNorm2d(torch.nn.Module):
    """
    BatchNorm2d where the batch statistics and the affine parameters are fixed.
    
    Copy-paste from torchvision.misc.ops with added eps before rsqrt,
    without which any other models than torchvision.models.resnet[18,34,50,101]
    produce nans.
    """

    def __init__(self, n):
        super(FrozenBatchNorm2d, self).__init__()
        self.register_buffer("weight", torch.ones(n))
        self.register_buffer("bias", torch.zeros(n))
        self.register_buffer("running_mean", torch.zeros(n))
        self.register_buffer("running_var", torch.ones(n))

    def _load_from_state_dict(self, state_dict, prefix, local_metadata, strict,
                              missing_keys, unexpected_keys, error_msgs):
        num_batches_tracked_key = prefix + 'num_batches_tracked'
        if num_batches_tracked_key in state_dict:
            del state_dict[num_batches_tracked_key]

        super(FrozenBatchNorm2d, self)._load_from_state_dict(
            state_dict, prefix, local_metadata, strict,
            missing_keys, unexpected_keys, error_msgs)

    def forward(self, x):
        # move reshapes to the beginning to make it user-friendly
        w = self.weight.reshape(1, -1, 1, 1)
        b = self.bias.reshape(1, -1, 1, 1)
        rv = self.running_var.reshape(1, -1, 1, 1)
        rm = self.running_mean.reshape(1, -1, 1, 1)
        eps = 1e-5
        scale = w * (rv + eps).rsqrt()
        bias = b - rm * scale
        return x * scale + bias

# ========================================
# 2. POSITION ENCODING (Official ACT - Sine only)
# ========================================

class PositionEmbeddingSine(nn.Module):
    """
    This is a more standard version of the position embedding, very similar to the one
    used by the Attention is all you need paper, generalized to work on images.
    """
    def __init__(self, num_pos_feats=64, temperature=10000, normalize=False, scale=None):
        super().__init__()
        self.num_pos_feats = num_pos_feats
        self.temperature = temperature
        self.normalize = normalize
        if scale is not None and normalize is False:
            raise ValueError("normalize should be True if scale is passed")
        if scale is None:
            scale = 2 * math.pi
        self.scale = scale

    def forward(self, tensor):
        """
        Args:
            tensor: [batch, channels, height, width]
        Returns:
            pos: [batch, num_pos_feats*2, height, width]
        """
        x = tensor
        # Create a mask of ones (ACT doesn't use actual masks for position encoding)
        not_mask = torch.ones_like(x[0, [0]])  # [1, height, width]
        
        y_embed = not_mask.cumsum(1, dtype=torch.float32)
        x_embed = not_mask.cumsum(2, dtype=torch.float32)
        
        if self.normalize:
            eps = 1e-6
            y_embed = y_embed / (y_embed[:, -1:, :] + eps) * self.scale
            x_embed = x_embed / (x_embed[:, :, -1:] + eps) * self.scale

        dim_t = torch.arange(self.num_pos_feats, dtype=torch.float32, device=x.device)
        dim_t = self.temperature ** (2 * (dim_t // 2) / self.num_pos_feats)

        pos_x = x_embed[:, :, :, None] / dim_t
        pos_y = y_embed[:, :, :, None] / dim_t
        pos_x = torch.stack((pos_x[:, :, :, 0::2].sin(), pos_x[:, :, :, 1::2].cos()), dim=4).flatten(3)
        pos_y = torch.stack((pos_y[:, :, :, 0::2].sin(), pos_y[:, :, :, 1::2].cos()), dim=4).flatten(3)
        pos = torch.cat((pos_y, pos_x), dim=3).permute(0, 3, 1, 2)
        return pos

def build_position_encoding(hidden_dim=256):
    """
    Official ACT position encoding builder
    Args:
        hidden_dim: transformer hidden dimension (default: 256)
    Returns:
        PositionEmbeddingSine object
    """
    N_steps = hidden_dim // 2  # 256 // 2 = 128
    position_embedding = PositionEmbeddingSine(N_steps, normalize=True)
    return position_embedding

# ========================================
# 3. BACKBONE BASE CLASS (Official ACT)
# ========================================

class BackboneBase(nn.Module):
    """Base class for backbones"""
    
    def __init__(self, backbone: nn.Module, train_backbone: bool, num_channels: int, return_interm_layers: bool):
        super().__init__()
        
        # ✅ ENABLE OFFICIAL LAYER FREEZING (matching official ACT)
        # This freezes early layers and only trains layer2, layer3, layer4
        for name, parameter in backbone.named_parameters():
            if not train_backbone or 'layer2' not in name and 'layer3' not in name and 'layer4' not in name:
                parameter.requires_grad_(False)
        
        if return_interm_layers:
            return_layers = {"layer1": "0", "layer2": "1", "layer3": "2", "layer4": "3"}
        else:
            return_layers = {'layer4': "0"}  # Only last layer (typical for ACT)
        
        # ✅ CRITICAL: Use IntermediateLayerGetter to extract specific layers
        self.body = IntermediateLayerGetter(backbone, return_layers=return_layers)
        self.num_channels = num_channels

    def forward(self, tensor):
        """
        Args:
            tensor: [batch, 3, height, width]
        Returns:
            xs: dict with extracted layer features
        """
        xs = self.body(tensor)
        return xs

# ========================================
# 4. BACKBONE CLASS (Official ACT)
# ========================================

class Backbone(BackboneBase):
    """ResNet backbone with frozen BatchNorm."""
    
    def __init__(self, name: str, train_backbone: bool, return_interm_layers: bool, dilation: bool):
        
        # ✅ CREATES THE ACTUAL RESNET MODEL
        # This is the core ResNet (resnet18, resnet34, resnet50, etc.)
        backbone = getattr(torchvision.models, name)( # Dynamically gets ResNet class (resnet34, resnet50, etc.)
            # DILATION CONTROL: Replaces stride with dilation in later layers for larger feature maps
            replace_stride_with_dilation=[False, False, dilation],  # Only apply dilation to layer4 if True
            # Controls whether to use dilated convolutions instead of stride for larger feature maps
            
            # PRETRAINED WEIGHTS: Official ACT uses is_main_process() for distributed training
            # Your version: pretrained=True (simpler, always load pretrained weights)
            # Official: pretrained=is_main_process() (only main process loads, then distributes)
            # pretrained=is_main_process(),  # Official version
            pretrained=True,  # Your simplified version
            
            # FROZEN BATCH NORM: Replace all BatchNorm2d with FrozenBatchNorm2d
            # This prevents batch statistics from updating during training
            # Keeps the pretrained statistics frozen for better stability, (keeps pretrained stats)
            norm_layer=FrozenBatchNorm2d
        )
        
        # ✅ SET OUTPUT CHANNELS BASED ON RESNET ARCHITECTURE
        # ResNet18/34: Final conv layer has 512 channels
        # ResNet50/101/152: Final conv layer has 2048 channels (due to bottleneck blocks)
        num_channels = 512 if name in ('resnet18', 'resnet34') else 2048
        
        # ✅ INITIALIZE PARENT CLASS (BackboneBase)
        # This sets up IntermediateLayerGetter and other backbone functionality
        super().__init__(backbone, train_backbone, num_channels, return_interm_layers)

# KEY DIFFERENCES IN YOUR VERSION:
# 1. pretrained=True vs pretrained=is_main_process()
#    - Your version is simpler and works fine for single-GPU training
#    - Official version is for distributed training across multiple GPUs

# 2. Both versions are functionally equivalent for our use case

# ========================================
# 5. JOINER CLASS (Official ACT - THE CRITICAL COMPONENT)
# ========================================

class Joiner(nn.Sequential):
    """
    Combines backbone and position encoding into a single module.
    This is the actual "backbone" used in ACT.
    """
    
    def __init__(self, backbone, position_embedding):
        super().__init__(backbone, position_embedding)
        # self[0] = backbone
        # self[1] = position_embedding

    def forward(self, tensor):
        """
        Args:
            tensor: [batch, 3, height, width]
        Returns:
            out: list of feature tensors
            pos: list of position encoding tensors
        """
        # ✅ STEP 1: Extract features using backbone
        xs = self[0](tensor)  # backbone forward pass
        
        # ✅ STEP 2: Generate position encodings for each feature level
        out = []
        pos = []
        for name, x in xs.items():
            out.append(x)  # Features: [batch, channels, H, W]
            # ✅ CRITICAL: Generate position encoding dynamically
            pos.append(self[1](x).to(x.dtype))  # Position: [batch, 256, H, W]
        
        return out, pos

# ========================================
# 6. BUILD BACKBONE (Official ACT)
# ========================================

def build_backbone(backbone_name='resnet34', hidden_dim=256, train_backbone=True, 
                  return_interm_layers=False, dilation=False):
    """
    Official ACT backbone builder
    
    Args:
        backbone_name: ResNet model name
        hidden_dim: transformer hidden dimension (for position encoding)
        train_backbone: whether to train backbone
        return_interm_layers: whether to return intermediate layers
        dilation: whether to use dilated convolutions
        
    Returns:
        Joiner object (backbone + position encoding combined)
    """
    # ✅ STEP 1: Build position encoding
    position_embedding = build_position_encoding(hidden_dim)
    # OFFICIAL: position_embedding = build_position_encoding(args)
    # YOURS:    position_embedding = build_position_encoding(hidden_dim)
    # COMMENT: Official passes entire args object, yours passes specific hidden_dim
    #          Both work, yours is more explicit about what's needed
    
    # ✅ STEP 2: Build backbone
    backbone = Backbone(backbone_name, train_backbone, return_interm_layers, dilation)
    # OFFICIAL: train_backbone = args.lr_backbone > 0
    # YOURS:    train_backbone=True (default parameter)
    # COMMENT: Official derives from learning rate, yours is explicit boolean
    #          Yours is simpler and clearer
    
    # OFFICIAL: return_interm_layers = args.masks
    # YOURS:    return_interm_layers=False (default parameter)
    # COMMENT: Official ties to mask usage, yours defaults to False
    #          For ACT, False is correct (only need final layer)
    
    # ✅ STEP 3: Create Joiner (combines backbone + position encoding)
    model = Joiner(backbone, position_embedding)
    
    # ✅ STEP 4: Set num_channels attribute (needed for input projection)
    model.num_channels = backbone.num_channels
    
    return model

# ========================================
# 7. OFFICIAL ACT VISUAL PROCESSING PIPELINE
# ========================================

def create_official_act_visual_pipeline(backbone_name='resnet34', hidden_dim=256):
    """
    Create the complete official ACT visual processing pipeline
    
    Returns:
        backbone: Joiner object (official ACT backbone)
        input_proj: Conv2d layer for feature projection
    """
    # ✅ STEP 1: Build official ACT backbone (Joiner)
    backbone = build_backbone(
        backbone_name=backbone_name,
        hidden_dim=hidden_dim,
        train_backbone=True,
        return_interm_layers=False,  # ACT typically uses only last layer
        dilation=False
    )
    
    # ✅ STEP 2: Create input projection (matches DETRVAE)
    # Projects backbone features to transformer hidden dimension
    input_proj = nn.Conv2d(backbone.num_channels, hidden_dim, kernel_size=1)
    
    print(f"✅ Official ACT Visual Pipeline Created:")
    print(f"   Backbone: {backbone_name} with FrozenBatchNorm2d")
    print(f"   Channels: {backbone.num_channels} → {hidden_dim}")
    print(f"   Position Encoding: Sine (normalize=True)")
    print(f"   Components: Joiner(Backbone + PositionEmbedding)")
    
    return backbone, input_proj

# ========================================
# 8. OFFICIAL ACT FEATURE EXTRACTION (Matches DETRVAE forward)
# ========================================

def extract_visual_features_official_act(backbone, input_proj, images, device):
    """
    Extract visual features exactly as done in DETRVAE.forward()
    
    This replicates lines 121-128 from the official code:
    features, pos = self.backbones[0](image[:, cam_id])
    features = features[0]  # take the last layer feature
    pos = pos[0]
    all_cam_features.append(self.input_proj(features))
    
    Args:
        backbone: Official ACT Joiner object
        input_proj: Conv2d projection layer
        images: [batch, 3, height, width] tensor
        device: torch device
        
    Returns:
        projected_features: [batch, hidden_dim, H, W] - ready for transformer
        pos: [batch, hidden_dim, H, W] - position encodings
    """
    backbone = backbone.to(device)
    input_proj = input_proj.to(device)
    backbone.train()
    input_proj.train()
    images = images.to(device)
    
   
    # ✅ OFFICIAL ACT: features, pos = self.backbones[0](image[:, cam_id])
    
    features_list, pos_list = backbone(images)
        
    # ✅ OFFICIAL ACT: features = features[0] (take the last layer feature)
    features = features_list[0]  # [batch, 512, H, W] for ResNet34
        
    # ✅ OFFICIAL ACT: pos = pos[0]
    pos = pos_list[0]  # [batch, 256, H, W] position encodings
        
    # ✅ OFFICIAL ACT: self.input_proj(features)
    projected_features = input_proj(features)  # [batch, 256, H, W]
    
    return projected_features, pos

# ========================================
# 9. MULTI-CAMERA PROCESSING (Official ACT Style)
# ========================================

def process_multi_camera_official_act(backbone, input_proj, images_dict, camera_names, device):
    """
    Process multiple cameras exactly as in DETRVAE.forward()
    
    This replicates the multi-camera loop:
    for cam_id, cam_name in enumerate(self.camera_names):
        features, pos = self.backbones[0](image[:, cam_id])
        features = features[0]
        pos = pos[0]
        all_cam_features.append(self.input_proj(features))
        all_cam_pos.append(pos)
    
    Args:
        backbone: Official ACT Joiner
        input_proj: Conv2d projection
        images_dict: dict with camera_name -> [batch, 3, H, W] tensors
        camera_names: list of camera names
        device: torch device
        
    Returns:
        src: [batch, hidden_dim, H, W*num_cameras] - concatenated features
        pos: [batch, hidden_dim, H, W*num_cameras] - concatenated positions
    """
    all_cam_features = []
    all_cam_pos = []
    
    for cam_id, cam_name in enumerate(camera_names):
        # Get images for this camera
        cam_images = images_dict[cam_name]  # [batch, 3, H, W]
        
        # ✅ OFFICIAL ACT PROCESSING
        projected_features, pos_encoding = extract_visual_features_official_act(
            backbone, input_proj, cam_images, device
        )
        
        all_cam_features.append(projected_features)  # [batch, 256, H, W]
        all_cam_pos.append(pos_encoding)            # [batch, 256, H, W]
    
    # ✅ OFFICIAL ACT: "fold camera dimension into width dimension"
    # src = torch.cat(all_cam_features, axis=3)
    # pos = torch.cat(all_cam_pos, axis=3)
    src = torch.cat(all_cam_features, dim=3)  # [batch, 256, H, W*num_cameras]
    pos = torch.cat(all_cam_pos, dim=3)       # [batch, 256, H, W*num_cameras]
    
    return src, pos

# ========================================
# 10. COMPLETE OFFICIAL ACT VISUAL SETUP
# ========================================

def setup_official_act_visual_system(backbone_name='resnet34', hidden_dim=256, device=None):
    """
    Setup the complete official ACT visual processing system
    
    Returns:
        backbone: Official ACT Joiner
        input_proj: Feature projection layer
        extract_fn: Function to extract features for single camera
        multi_cam_fn: Function to process multiple cameras
    """
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    # Create official ACT components
    backbone, input_proj = create_official_act_visual_pipeline(backbone_name, hidden_dim)
    
    # Move to device
    backbone = backbone.to(device)
    input_proj = input_proj.to(device)
    
    # Create processing functions
    def extract_single_camera(images):
        return extract_visual_features_official_act(backbone, input_proj, images, device)
    
    def process_multiple_cameras(images_dict, camera_names):
        return process_multi_camera_official_act(backbone, input_proj, images_dict, camera_names, device)
    
    print(f"✅ Official ACT Visual System Ready on {device}")
    print(f"   Ready for: features, pos = backbone(images)")
    print(f"   Ready for: projected_features = input_proj(features)")
    print(f"   Ready for: multi-camera concatenation")
    
    return backbone, input_proj, extract_single_camera, process_multiple_cameras


print("✅ Official ACT Visual Pipeline Ready!")

## CVAE + transformer (extract style variable z )

In [ ]:
# ========================================
# CELL 1: OFFICIAL ACT CVAE MODEL (Clean Version)
# ========================================

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from torch.autograd import Variable
from torchvision import transforms

# Import the official transformer components
import sys
sys.path.append(r'C:\Users\Administrator\Documents\transformer\ACT-Shaka\detr\models')
from transformer import build_transformer, TransformerEncoder, TransformerEncoderLayer

# ========================================
# HELPER FUNCTIONS (Official ACT)
# ========================================

def reparametrize(mu, logvar):
    """Official VAE reparameterization trick from detr_vae.py"""
    std = logvar.div(2).exp()                           # Convert log(σ²) to σ
    eps = Variable(std.data.new(std.size()).normal_())  # Sample ε ~ N(0,1)
    return mu + std * eps                               # z = μ + σε

def get_sinusoid_encoding_table(n_position, d_hid):
    """Official sinusoidal position encoding from detr_vae.py"""
    def get_position_angle_vec(position):
        return [position / np.power(10000, 2 * (hid_j // 2) / d_hid) for hid_j in range(d_hid)]

    sinusoid_table = np.array([get_position_angle_vec(pos_i) for pos_i in range(n_position)])
    sinusoid_table[:, 0::2] = np.sin(sinusoid_table[:, 0::2])  # Even positions: sin
    sinusoid_table[:, 1::2] = np.cos(sinusoid_table[:, 1::2])  # Odd positions: cos

    return torch.FloatTensor(sinusoid_table).unsqueeze(0)

def build_encoder(args):
    """Official CVAE encoder builder from detr_vae.py"""
    encoder_layer = TransformerEncoderLayer(
        args.hidden_dim, args.nheads, args.dim_feedforward,
        args.dropout, "relu", args.pre_norm
    )
    encoder_norm = nn.LayerNorm(args.hidden_dim) if args.pre_norm else None
    encoder = TransformerEncoder(encoder_layer, args.enc_layers, encoder_norm)
    return encoder

class TransformerArgs:
    """Configuration object for official build functions"""
    def __init__(self, hidden_dim=256, dropout=0.1, nheads=8, dim_feedforward=2048, 
                 enc_layers=4, dec_layers=6, pre_norm=False, num_queries=10, camera_names=None):
        self.hidden_dim = hidden_dim
        self.dropout = dropout
        self.nheads = nheads
        self.dim_feedforward = dim_feedforward
        self.enc_layers = enc_layers
        self.dec_layers = dec_layers
        self.pre_norm = pre_norm
        self.num_queries = num_queries
        self.camera_names = camera_names or ['camera_0']

# ========================================
# MAIN ACT MODEL (Official DETRVAE)
# ========================================

class DETRVAE_Official(nn.Module):
    """
    Official ACT model from detr_vae.py
    Modified to accept pre-extracted visual features from your backbone
    """
    
    def __init__(self, transformer, encoder, state_dim, num_queries, camera_names, hidden_dim):
        super().__init__()
        
        # Core model components
        self.num_queries = num_queries          # Action chunk size (10)
        self.camera_names = camera_names        # Camera names
        self.transformer = transformer          # Main transformer decoder
        self.encoder = encoder                  # CVAE encoder
        
        # Output prediction heads
        self.action_head = nn.Linear(hidden_dim, state_dim)     # Actions: [256] → [6]
        self.is_pad_head = nn.Linear(hidden_dim, 1)             # Padding: [256] → [1]
        self.query_embed = nn.Embedding(num_queries, hidden_dim)  # Action queries: [10, 256]
        
        # Robot state projection
        self.input_proj_robot_state = nn.Linear(6, hidden_dim)  # QPos: [6] → [256]
        
        # CVAE encoder components
        self.latent_dim = 64                                     # Latent dimension
        self.cls_embed = nn.Embedding(1, hidden_dim)            # CLS token: [1, 256]
        self.encoder_action_proj = nn.Linear(6, hidden_dim)     # Action proj: [6] → [256]
        self.encoder_joint_proj = nn.Linear(6, hidden_dim)      # QPos proj: [6] → [256]
        self.latent_proj = nn.Linear(hidden_dim, self.latent_dim * 2)  # VAE params: [256] → [64]
        
        # Position encoding for CVAE sequence: [CLS, qpos, action_1, ..., action_10]
        self.register_buffer('pos_table', get_sinusoid_encoding_table(1 + 1 + num_queries, hidden_dim))
        
        # Decoder components
        self.latent_out_proj = nn.Linear(self.latent_dim, hidden_dim)  # Latent: [32] → [256]
        self.additional_pos_embed = nn.Embedding(2, hidden_dim)        # Additional pos: [2, 256]
        
    def forward(self, qpos, src, pos, actions=None, is_pad=None):
        """
        Forward pass through ACT model
        
        Args:
            qpos: [batch, 6] - Current robot joint positions
            src: [batch, 256, H, W] - Pre-extracted visual features from YOUR backbone
            pos: [batch, 256, H, W] - Pre-extracted position embeddings from YOUR backbone
            actions: [batch, 10, 6] - Demonstrated actions (training only)
            is_pad: [batch, 10] - Padding mask (training only)
            
        Returns:
            a_hat: [batch, 10, 6] - Predicted actions
            is_pad_hat: [batch, 10, 1] - Predicted padding
            [mu, logvar]: VAE parameters for loss computation
        """
        is_training = actions is not None  # Check if we're in training mode
        bs, _ = qpos.shape                 # Get batch size
        
        # ========================================
        # STEP 1: CVAE ENCODING (Training only)
        # ========================================
        if is_training:
            # Project inputs to embedding space
            action_embed = self.encoder_action_proj(actions)      # [bs, 10, 256]
            qpos_embed = self.encoder_joint_proj(qpos)            # [bs, 256]
            qpos_embed = torch.unsqueeze(qpos_embed, axis=1)      # [bs, 1, 256]
            
            # Get CLS token and replicate for batch
            cls_embed = self.cls_embed.weight                     # [1, 256]
            cls_embed = torch.unsqueeze(cls_embed, axis=0).repeat(bs, 1, 1)  # [bs, 1, 256]
            
            # Create encoder input sequence: [CLS, qpos, action_1, ..., action_10]
            encoder_input = torch.cat([cls_embed, qpos_embed, action_embed], axis=1)  # [bs, 12, 256]
            encoder_input = encoder_input.permute(1, 0, 2)       # [12, bs, 256] (transformer format)
            
            # Create padding mask for encoder (CLS and qpos are never padded)
            cls_joint_is_pad = torch.full((bs, 2), False).to(qpos.device)  # [bs, 2]
            full_is_pad = torch.cat([cls_joint_is_pad, is_pad], axis=1)    # [bs, 12]
            
            # Add positional encoding
            pos_embed = self.pos_table.clone().detach()          # [1, 12, 256]
            pos_embed = pos_embed.permute(1, 0, 2)               # [12, 1, 256]
            
            # CVAE encoder forward pass
            encoder_output = self.encoder(encoder_input, pos=pos_embed, src_key_padding_mask=full_is_pad)
            encoder_output = encoder_output[0]                   # [bs, 256] (CLS output only)
            
            # Extract VAE parameters from CLS output
            latent_info = self.latent_proj(encoder_output)       # [bs, 64]
            mu = latent_info[:, :self.latent_dim]                # [bs, 32] Mean
            logvar = latent_info[:, self.latent_dim:]            # [bs, 32] Log variance
            latent_sample = reparametrize(mu, logvar)            # [bs, 32] Sample z
            latent_input = self.latent_out_proj(latent_sample)   # [bs, 256] Project back
        else:
            # Inference mode: use zero latent
            mu = logvar = None
            latent_sample = torch.zeros([bs, self.latent_dim], dtype=torch.float32).to(qpos.device)
            latent_input = self.latent_out_proj(latent_sample)   # [bs, 256]
        
        # ========================================
        # STEP 2: PREPARE INPUTS FOR TRANSFORMER DECODER
        # ========================================
        # Project robot state to embedding space
        proprio_input = self.input_proj_robot_state(qpos)        # [bs, 256]
        
        # ========================================
        # STEP 3: TRANSFORMER DECODER FORWARD PASS
        # ========================================
        # This is where the magic happens! The transformer combines:
        # - Visual features (src) from your backbone
        # - Position encodings (pos) from your backbone  
        # - Latent variable (latent_input) from CVAE
        # - Robot state (proprio_input) 
        # - Action queries (self.query_embed.weight)
        hs = self.transformer(
            src,                                    # [bs, 256, H, W] Visual features
            None,                                   # No attention mask
            self.query_embed.weight,                # [10, 256] Action queries
            pos,                                    # [bs, 256, H, W] Position encodings
            latent_input,                           # [bs, 256] Latent variable
            proprio_input,                          # [bs, 256] Robot state
            self.additional_pos_embed.weight        # [2, 256] Additional positions
        )[0]                                        # Take final layer output
        
        # ========================================
        # STEP 4: GENERATE PREDICTIONS
        # ========================================
        a_hat = self.action_head(hs)               # [bs, 10, 6] Predicted actions
        is_pad_hat = self.is_pad_head(hs)          # [bs, 10, 1] Predicted padding
        
        return a_hat, is_pad_hat, [mu, logvar]

def build_act_model_with_visual_features(state_dim=6, num_queries=10, hidden_dim=256,
                                        camera_names=None, device=None):
    """
    Build complete ACT model that works with your pre-extracted visual features
    
    This function creates the official ACT model but skips the visual backbone
    since you already have visual features from your earlier cells.
    """
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    # Create configuration for official build functions
    args = TransformerArgs(
        hidden_dim=hidden_dim,
        dropout=0.1,
        nheads=8,
        dim_feedforward=2048,
        enc_layers=4,                # CVAE encoder layers
        dec_layers=6,                # Transformer decoder layers
        pre_norm=False,
        num_queries=num_queries,
        camera_names=camera_names
    )
    
    # Build official transformer components
    transformer = build_transformer(args)      # Main transformer
    encoder = build_encoder(args)              # CVAE encoder
    
    # Create complete model
    model = DETRVAE_Official(
        transformer=transformer,
        encoder=encoder,
        state_dim=state_dim,
        num_queries=num_queries,
        camera_names=camera_names or ['camera_0'],
        hidden_dim=hidden_dim
    )
    
    model = model.to(device)
    n_parameters = sum(p.numel() for p in model.parameters() if p.requires_grad)
    
    print(f"✅ ACT Model Created:")
    print(f"   Parameters: {n_parameters/1e6:.2f}M")
    print(f"   State dim: {state_dim}")
    print(f"   Action chunk: {num_queries}")
    print(f"   Device: {device}")
    
    return model

print("✅ ACT Model Implementation Complete!")

# training phase

In [ ]:


# ========================================
# CELL 2: TRAINING COMPONENTS (Clean Version)
# ========================================

import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import os
from copy import deepcopy


def create_official_act_optimizer(policy, backbone, input_proj, lr=5e-5, lr_backbone=1e-5, weight_decay=1e-4):
    """Official ACT optimizer pattern from main.py"""
    
    # Combine all named parameters (simulating single model)
    all_named_params = []
    for name, param in policy.named_parameters():
        all_named_params.append((f"policy.{name}", param))
    for name, param in input_proj.named_parameters():
        all_named_params.append((f"input_proj.{name}", param))
    for name, param in backbone.named_parameters():
        all_named_params.append((f"backbone.{name}", param))
    
    # Official ACT parameter grouping
    param_dicts = [
        {"params": [p for n, p in all_named_params if "backbone" not in n and p.requires_grad]},
        {
            "params": [p for n, p in all_named_params if "backbone" in n and p.requires_grad],
            "lr": lr_backbone,  # Lower LR for backbone
        },
    ]
    
    optimizer = torch.optim.AdamW(param_dicts, lr=lr, weight_decay=weight_decay)
    return optimizer

# ========================================
# OFFICIAL LOSS FUNCTIONS
# ========================================

def kl_divergence(mu, logvar):
    """Official KL divergence from training/policy.py"""
    batch_size = mu.size(0)
    assert batch_size != 0
    
    # Handle different tensor dimensions
    if mu.data.ndimension() == 4:
        mu = mu.view(mu.size(0), mu.size(1))
    if logvar.data.ndimension() == 4:
        logvar = logvar.view(logvar.size(0), logvar.size(1))

    # KL divergence formula: -0.5 * Σ(1 + log(σ²) - μ² - σ²)
    klds = -0.5 * (1 + logvar - mu.pow(2) - logvar.exp())
    total_kld = klds.sum(1).mean(0, True)           # Sum over latent dims, mean over batch
    dimension_wise_kld = klds.mean(0)               # Mean over batch for each dimension
    mean_kld = klds.mean(1).mean(0, True)           # Mean over everything

    return total_kld, dimension_wise_kld, mean_kld

def compute_act_loss(predictions, targets, mu, logvar, is_pad, kl_weight=100):
    """
    Official ACT loss computation from training/policy.py
    
    Args:
        predictions: [batch, chunk_size, 6] - Predicted actions
        targets: [batch, chunk_size, 6] - Ground truth actions
        mu: [batch, latent_dim] - VAE mean parameters
        logvar: [batch, latent_dim] - VAE log variance parameters
        is_pad: [batch, chunk_size] - Padding mask (True = padded)
        kl_weight: KL divergence weight (100 in official config)
    
    Returns:
        loss_dict: Dictionary with loss components
    """
    # Compute KL divergence (only during training when mu/logvar exist)
    if mu is not None and logvar is not None:
        total_kld, _, _ = kl_divergence(mu, logvar)
        kl_loss = total_kld[0]  # Extract scalar
    else:
        kl_loss = torch.tensor(0.0, device=predictions.device)
    
    # Compute L1 reconstruction loss with padding mask
    all_l1 = F.l1_loss(targets, predictions, reduction='none')  # [batch, chunk, 6]
    l1_loss = (all_l1 * ~is_pad.unsqueeze(-1)).mean()          # Apply mask and average
    
    # Combine losses
    total_loss = l1_loss + kl_loss * kl_weight
    
    return {
        'l1': l1_loss,
        'kl': kl_loss,
        'loss': total_loss
    }

# ========================================
# FORWARD PASS FUNCTION
# ========================================

def forward_pass(model, qpos_data, image_data, action_data, is_pad, 
                backbone, input_proj, device):
    """
    Complete forward pass through ACT model with loss computation
    
    This function:
    1. Normalizes images (ImageNet stats)
    2. Extracts visual features using your backbone
    3. Runs ACT model forward pass
    4. Computes loss if training data provided
    
    Args:
        model: ACT model
        qpos_data: [batch, 6] robot positions
        image_data: [batch, 3, H, W] camera images
        action_data: [batch, chunk_size, 6] actions
        is_pad: [batch, chunk_size] padding mask
        backbone: Your visual backbone from earlier cells
        input_proj: Input projection layer from earlier cells
        device: torch device
    
    Returns:
        loss_dict: Dictionary with losses and predictions
    """
    # Move data to device
    qpos_data = qpos_data.to(device)
    image_data = image_data.to(device)
    action_data = action_data.to(device)
    is_pad = is_pad.to(device)
    
    # Official ACT image normalization (ImageNet statistics)
    normalize = transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
    image_data = normalize(image_data)
    
    # Truncate actions to model's chunk size
    num_queries = model.num_queries
    action_data = action_data[:, :num_queries]
    is_pad = is_pad[:, :num_queries]
    
    # Extract visual features using your backbone from earlier cells
    projected_features, pos_encodings = extract_visual_features_official_act(
        backbone, input_proj, image_data, device
    )
    
    # ACT model forward pass
    a_hat, is_pad_hat, (mu, logvar) = model(
        qpos=qpos_data,
        src=projected_features,      # Your visual features
        pos=pos_encodings,           # Your position encodings
        actions=action_data,         # Ground truth actions (training)
        is_pad=is_pad               # Padding mask (training)
    )
    
    # Compute loss
    loss_dict = compute_act_loss(
        predictions=a_hat,
        targets=action_data,
        mu=mu,
        logvar=logvar,
        is_pad=is_pad,
        kl_weight=100  # From official config
    )
    
    # Add predictions for analysis
    loss_dict['predictions'] = a_hat
    loss_dict['is_pad_pred'] = is_pad_hat
    
    return loss_dict

# ========================================
# TRAINING UTILITIES
# ========================================

def set_seed(seed):
    torch.manual_seed(seed)
    np.random.seed(seed)

def compute_dict_mean(epoch_dicts):
    """Average loss dictionaries across batches"""
    result = {}
    for key in epoch_dicts[0].keys():
        if key in ['predictions', 'is_pad_pred']:
            continue  # Skip non-scalar values
        result[key] = torch.stack([d[key] for d in epoch_dicts]).mean()
    return result

def detach_dict(d):
    """Detach tensors in dictionary for storage"""
    new_d = {}
    for k, v in d.items():
        if isinstance(v, torch.Tensor):
            new_d[k] = v.detach()
        else:
            new_d[k] = v
    return new_d

def create_optimizer_and_scheduler(model, lr=5e-5, weight_decay=1e-4):
    """Create optimizer and scheduler matching official config"""
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', patience=200, factor=0.5, verbose=True
    )
    return optimizer, scheduler

print("✅ Training Components Ready!")
print("📝 Key Functions:")
print("   - forward_pass(): Complete ACT forward pass with loss")
print("   - compute_act_loss(): Official ACT loss computation")
print("   - create_optimizer_and_scheduler(): Training setup")

In [ ]:
# ========================================
# CORRECTED: Use Official ACTPolicy Structure
# ========================================

def create_official_act_policy(state_dim=6, num_queries=100, hidden_dim=512, 
                              camera_names=['front'], kl_weight=100, device=None):
    """
    Create ACTPolicy exactly like the official repository
    """
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    # ✅ OFFICIAL ARGS OVERRIDE (matches your config.py)
    args_override = {
        'policy_class': 'ACT',
        'kl_weight': kl_weight,
        'chunk_size': num_queries,
        'hidden_dim': hidden_dim,
        'dim_feedforward': 2048,
        'lr': 5e-5,
        'backbone': 'resnet34',
        'loss_function': 'l1',
        'seed': 42,
        'enc_layers': 4,
        'dec_layers': 6,
        'nheads': 8,
        'camera_names': camera_names,
        'state_dim': state_dim,
        'lr_backbone': 1e-5,
        'weight_decay': 1e-4,
        'dropout': 0.1,
        'pre_norm': False,
        'num_queries': num_queries,
    }
    
    # ✅ IMPORT OFFICIAL POLICY CLASS
    import sys
    sys.path.append(r'C:\Users\Administrator\Documents\transformer\ACT-Shaka\training')
    from policy import ACTPolicy
    
    # ✅ CREATE OFFICIAL ACT POLICY (this calls build_ACT_model_and_optimizer internally)
    policy = ACTPolicy(args_override)
    policy = policy.to(device)
    
    # Count parameters
    total_params = sum(p.numel() for p in policy.parameters() if p.requires_grad)
    model_params = sum(p.numel() for p in policy.model.parameters() if p.requires_grad)
    
    print(f"✅ Official ACTPolicy Created:")
    print(f"   Total parameters: {total_params/1e6:.2f}M")
    print(f"   Model parameters: {model_params/1e6:.2f}M")
    print(f"   KL weight: {policy.kl_weight}")
    print(f"   Queries: {policy.model.num_queries}")
    print(f"   Compatible with official loading: ✅")
    
    return policy

# ========================================
# OFFICIAL TRAINING FUNCTION (Fixed)
# ========================================

def train_act_official_corrected(train_dataloader, val_dataloader, checkpoint_dir, task='task1c'):
    """
    Training function matching official train.py exactly
    """
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    # ✅ CREATE OFFICIAL ACT POLICY (matches official structure)
    policy = create_official_act_policy(
        state_dim=6,
        num_queries=100,  # From POLICY_CONFIG
        hidden_dim=512,   # From POLICY_CONFIG
        camera_names=['front'],
        kl_weight=100,
        device=device
    )
    
    # ✅ USE OFFICIAL OPTIMIZER (already created inside ACTPolicy)
    optimizer = policy.optimizer
    
    # Add scheduler
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, 'min', patience=200, factor=0.5)
    
    os.makedirs(checkpoint_dir, exist_ok=True)
    
    train_history = []
    validation_history = []
    min_val_loss = np.inf
    best_ckpt_info = None
    
    num_epochs = 5000
    
    for epoch in range(num_epochs):
        print(f'\nEpoch {epoch}')
        
        # ========================================
        # VALIDATION PHASE (Official train.py structure)
        # ========================================
        with torch.inference_mode():
            policy.eval()
            epoch_dicts = []
            
            for batch_idx, data in enumerate(val_dataloader):
                # ✅ OFFICIAL FORWARD PASS (matches train.py line 17)
                forward_dict = forward_pass_official_style(data, policy, device)
                epoch_dicts.append(forward_dict)
            
            epoch_summary = compute_dict_mean(epoch_dicts)
            validation_history.append(epoch_summary)
            
            epoch_val_loss = epoch_summary['loss']
            scheduler.step(epoch_val_loss)
            
            # Learning rate tracking
            for param_group in optimizer.param_groups:
                current_lr = param_group['lr']
            
            if not hasattr(scheduler, 'last_lr'):
                scheduler.last_lr = current_lr
            
            if current_lr != scheduler.last_lr:
                print(f"Learning rate changed from {scheduler.last_lr:.2e} to {current_lr:.2e}")
                scheduler.last_lr = current_lr
            
            if epoch_val_loss < min_val_loss:
                min_val_loss = epoch_val_loss
                best_ckpt_info = (epoch, min_val_loss, deepcopy(policy.state_dict()))
        
        print(f'Val loss:   {epoch_val_loss:.5f}')
        summary_string = ''
        for k, v in epoch_summary.items():
            summary_string += f'{k}: {v.item():.3f} '
        print(summary_string)
        
        # ========================================
        # TRAINING PHASE (Official train.py structure)
        # ========================================
        policy.train()
        optimizer.zero_grad()
        train_epoch_dicts = []
        
        for batch_idx, data in enumerate(train_dataloader):
            # ✅ OFFICIAL FORWARD PASS
            forward_dict = forward_pass_official_style(data, policy, device)
            
            # Backward pass (matches train.py lines 79-83)
            loss = forward_dict['loss']
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            
            train_epoch_dicts.append(detach_dict(forward_dict))
        
        # Compute training summary
        epoch_summary = compute_dict_mean(train_epoch_dicts)
        epoch_train_loss = epoch_summary['loss']
        
        print(f'Train loss: {epoch_train_loss:.5f}')
        summary_string = ''
        for k, v in epoch_summary.items():
            summary_string += f'{k}: {v.item():.3f} '
        print(summary_string)
        
        # ========================================
        # ✅ OFFICIAL CHECKPOINT SAVING (matches train.py line 105)
        # ========================================
        save_freq = 50 if epoch < 500 else 500
        if epoch % save_freq == 0:
            ckpt_path = os.path.join(checkpoint_dir, f"policy_epoch_{epoch}_seed_42.ckpt")
            torch.save(policy.state_dict(), ckpt_path)  # ✅ Official style
            
            # Verify checkpoint size
            checkpoint_size_mb = os.path.getsize(ckpt_path) / (1024 * 1024)
            print(f"Checkpoint saved: {ckpt_path}")
            print(f"Checkpoint size: {checkpoint_size_mb:.1f} MB")
    
    # ✅ SAVE FINAL CHECKPOINT (matches train.py line 107)
    ckpt_path = os.path.join(checkpoint_dir, f'policy_last.ckpt')
    torch.save(policy.state_dict(), ckpt_path)
    
    # ✅ SAVE BEST CHECKPOINT (matches train.py lines 111-114)
    if best_ckpt_info is not None:
        best_epoch, best_val_loss, best_state_dict = best_ckpt_info
        best_ckpt_path = os.path.join(checkpoint_dir, f"policy_best_epoch_{best_epoch}_val_{best_val_loss:.4f}.ckpt")
        torch.save(best_state_dict, best_ckpt_path)
        
        best_size_mb = os.path.getsize(best_ckpt_path) / (1024 * 1024)
        print(f"✅ Best checkpoint saved: {best_ckpt_path}")
        print(f"Size: {best_size_mb:.1f} MB")
        print(f"Expected: ~350-400 MB (matching official ACT)")
    
    return policy, {'train': train_history, 'val': validation_history}

def forward_pass_official_style(data, policy, device):
    """
    Forward pass matching official train.py line 17-21
    """
    image_data, qpos_data, action_data, is_pad = data
    
    # Move to device (matches train.py line 18)
    image_data = image_data.to(device)
    qpos_data = qpos_data.to(device)
    action_data = action_data.to(device)
    is_pad = is_pad.to(device)
    
    # ✅ OFFICIAL FORWARD PASS (matches train.py line 19)
    # This calls ACTPolicy.__call__() which handles normalization internally
    return policy(qpos_data, image_data, action_data, is_pad)

In [ ]:
# ========================================
# CORRECTED TRAINING SETUP AND EXECUTION
# ========================================

import torch
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import os
import pickle
import numpy as np
import matplotlib.pyplot as plt
from copy import deepcopy
from tqdm import tqdm

# ========================================
# MISSING UTILITY FUNCTIONS (Essential)
# ========================================

def set_seed(seed):
    """Set random seeds for reproducibility"""
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

def compute_dict_mean(epoch_dicts):
    """Average loss dictionaries across batches"""
    result = {}
    for key in epoch_dicts[0].keys():
        if key in ['predictions', 'is_pad_pred']:
            continue  # Skip non-scalar values
        result[key] = torch.stack([d[key] for d in epoch_dicts]).mean()
    return result

def detach_dict(d):
    """Detach tensors in dictionary for storage"""
    new_d = {}
    for k, v in d.items():
        if isinstance(v, torch.Tensor):
            new_d[k] = v.detach()
        else:
            new_d[k] = v
    return new_d

# ========================================
# FIXED DATASET CLASS (Consistent Chunk Size)
# ========================================

class ACTDataset(Dataset):
    """
    Dataset class matching official ACT training approach
    """
    def __init__(self, qpos_df, action_df, image_dict_list, chunk_size=100):  # ✅ Fixed to 100
        self.qpos_df = qpos_df
        self.action_df = action_df
        self.image_dict_list = image_dict_list
        self.chunk_size = chunk_size
        
        # Total timesteps across ALL episodes
        self.total_timesteps = len(qpos_df)
        
        # Create flattened image array
        self.all_images = []
        for episode_dict in image_dict_list:
            episode_images = episode_dict['front']  # [timesteps, H, W, 3]
            self.all_images.extend(episode_images)
        
        self.all_images = np.array(self.all_images)  # [total_timesteps, H, W, 3]
        
        print(f"📊 Dataset Statistics:")
        print(f"   Total timesteps: {self.total_timesteps}")
        print(f"   Images shape: {self.all_images.shape}")
        print(f"   Chunk size: {chunk_size}")
        
        # Verify data alignment
        assert len(self.all_images) == self.total_timesteps, \
            f"Image count {len(self.all_images)} != timestep count {self.total_timesteps}"
    
    def __len__(self):
        return self.total_timesteps
    
    def __getitem__(self, idx):
        # Get current qpos
        qpos = torch.FloatTensor(self.qpos_df.iloc[idx, 1:].values)  # Skip timestamp
        
        # Get current image
        if idx < len(self.all_images):
            image = self.all_images[idx]  # [H, W, 3] uint8
            image = torch.FloatTensor(image).permute(2, 0, 1) / 255.0  # [3, H, W]
        else:
            image = torch.zeros(3, 480, 640)
        
        # Get action chunk with padding
        action_chunk = torch.zeros(self.chunk_size, 6)
        is_pad = torch.ones(self.chunk_size, dtype=torch.bool)
        
        end_idx = min(idx + self.chunk_size, self.total_timesteps)
        available_actions = end_idx - idx
        
        if available_actions > 0:
            actions_data = self.action_df.iloc[idx:end_idx, 1:].values
            action_chunk[:available_actions] = torch.FloatTensor(actions_data)
            is_pad[:available_actions] = False
        
        return image, qpos, action_chunk, is_pad

# ========================================
# CORRECTED TRAINING FUNCTION (Official ACT Style)
# ========================================

def train_act_official_corrected(train_dataloader, val_dataloader, checkpoint_dir, task='task1c'):
    """
    Training function matching official train.py exactly
    """
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    # ✅ CREATE OFFICIAL ACT POLICY (Consistent Configuration)
    policy = create_official_act_policy(
        state_dim=6,
        num_queries=100,    # ✅ Consistent with official POLICY_CONFIG
        hidden_dim=512,     # ✅ Consistent with official POLICY_CONFIG
        camera_names=['front'],
        kl_weight=100,
        device=device
    )
    
    # ✅ USE OFFICIAL OPTIMIZER (already created inside ACTPolicy)
    optimizer = policy.optimizer
    
    # Add scheduler
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, 'min', patience=200, factor=0.5)
    
    os.makedirs(checkpoint_dir, exist_ok=True)
    
    train_history = []
    validation_history = []
    min_val_loss = np.inf
    best_ckpt_info = None
    
    num_epochs = 5000  # From official config
    
    for epoch in range(num_epochs):
        print(f'\nEpoch {epoch}')
        
        # ========================================
        # VALIDATION PHASE
        # ========================================
        with torch.inference_mode():
            policy.eval()
            epoch_dicts = []
            
            for batch_idx, data in enumerate(val_dataloader):
                # ✅ OFFICIAL FORWARD PASS
                forward_dict = forward_pass_official_style(data, policy, device)
                epoch_dicts.append(forward_dict)
            
            epoch_summary = compute_dict_mean(epoch_dicts)
            validation_history.append(epoch_summary)
            
            epoch_val_loss = epoch_summary['loss']
            scheduler.step(epoch_val_loss)
            
            if epoch_val_loss < min_val_loss:
                min_val_loss = epoch_val_loss
                best_ckpt_info = (epoch, min_val_loss, deepcopy(policy.state_dict()))
        
        print(f'Val loss:   {epoch_val_loss:.5f}')
        summary_string = ''
        for k, v in epoch_summary.items():
            summary_string += f'{k}: {v.item():.3f} '
        print(summary_string)
        
        # ========================================
        # TRAINING PHASE
        # ========================================
        policy.train()
        optimizer.zero_grad()
        train_epoch_dicts = []
        
        for batch_idx, data in enumerate(train_dataloader):
            # ✅ OFFICIAL FORWARD PASS
            forward_dict = forward_pass_official_style(data, policy, device)
            
            # Backward pass
            loss = forward_dict['loss']
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            
            train_epoch_dicts.append(detach_dict(forward_dict))
        
        # Compute training summary
        epoch_summary = compute_dict_mean(train_epoch_dicts)
        epoch_train_loss = epoch_summary['loss']
        train_history.append(epoch_summary)
        
        print(f'Train loss: {epoch_train_loss:.5f}')
        summary_string = ''
        for k, v in epoch_summary.items():
            summary_string += f'{k}: {v.item():.3f} '
        print(summary_string)
        
        # ========================================
        # ✅ OFFICIAL CHECKPOINT SAVING
        # ========================================
        save_freq = 50 if epoch < 500 else 500
        if epoch % save_freq == 0:
            ckpt_path = os.path.join(checkpoint_dir, f"policy_epoch_{epoch}_seed_42.ckpt")
            torch.save(policy.state_dict(), ckpt_path)  # ✅ Official style
            
            # Verify checkpoint size
            checkpoint_size_mb = os.path.getsize(ckpt_path) / (1024 * 1024)
            print(f"Checkpoint saved: {ckpt_path}")
            print(f"Checkpoint size: {checkpoint_size_mb:.1f} MB")
    
    # ✅ SAVE FINAL CHECKPOINT
    ckpt_path = os.path.join(checkpoint_dir, f'policy_last.ckpt')
    torch.save(policy.state_dict(), ckpt_path)
    
    # ✅ SAVE BEST CHECKPOINT
    if best_ckpt_info is not None:
        best_epoch, best_val_loss, best_state_dict = best_ckpt_info
        best_ckpt_path = os.path.join(checkpoint_dir, f"policy_best_epoch_{best_epoch}_val_{best_val_loss:.4f}.ckpt")
        torch.save(best_state_dict, best_ckpt_path)
        
        best_size_mb = os.path.getsize(best_ckpt_path) / (1024 * 1024)
        print(f"✅ Best checkpoint saved: {best_ckpt_path}")
        print(f"Size: {best_size_mb:.1f} MB")
        print(f"Expected: ~350-400 MB (matching official ACT)")
    
    return policy, {'train': train_history, 'val': validation_history}

def forward_pass_official_style(data, policy, device):
    """
    Forward pass matching official train.py
    """
    image_data, qpos_data, action_data, is_pad = data
    
    # Move to device
    image_data = image_data.to(device)
    qpos_data = qpos_data.to(device)
    action_data = action_data.to(device)
    is_pad = is_pad.to(device)
    
    # ✅ OFFICIAL FORWARD PASS (ACTPolicy handles everything internally)
    return policy(qpos_data, image_data, action_data, is_pad)

# ========================================
# COMPLETE TRAINING SETUP AND EXECUTION
# ========================================

def setup_and_train_act():
    """
    Complete setup and training pipeline
    """
    # Set seed for reproducibility
    set_seed(42)
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"🚀 Training on device: {device}")
    
    # ========================================
    # STEP 1: Create Dataset
    # ========================================
    print("📊 Creating dataset...")
    chunk_size = 100  # ✅ Consistent with official ACT
    
    dataset = ACTDataset(
        qpos_df=qpos_df,
        action_df=action_df,
        image_dict_list=image_dict_list,
        chunk_size=chunk_size
    )
    
    print(f"Dataset created with {len(dataset)} samples")
    
    # ========================================
    # STEP 2: Create Train/Val Split
    # ========================================
    train_ratio = 0.8
    train_size = int(len(dataset) * train_ratio)
    val_size = len(dataset) - train_size
    
    train_dataset, val_dataset = torch.utils.data.random_split(dataset, [train_size, val_size])
    
    # Create data loaders
    batch_size_train = 8   # Reduced for memory with chunk_size=100
    batch_size_val = 8
    
    train_dataloader = DataLoader(train_dataset, batch_size=batch_size_train, shuffle=True)
    val_dataloader = DataLoader(val_dataset, batch_size=batch_size_val, shuffle=False)
    
    print(f"Train samples: {len(train_dataset)}, Val samples: {len(val_dataset)}")
    
    # ========================================
    # STEP 3: Setup Checkpoint Directory
    # ========================================
    task = 'task1c'
    checkpoint_dir = os.path.join(r'C:\Users\Administrator\Documents\transformer\ACT-Shaka\checkpoints_nb', task)
    
    print(f"📁 Checkpoint directory: {checkpoint_dir}")
    
    # ========================================
    # STEP 4: Start Training
    # ========================================
    print("🚀 Starting training...")
    print("📝 OFFICIAL ACT TRAINING:")
    print("   - Continuous timeline (no episode boundaries)")
    print("   - Padding for incomplete chunks")
    print("   - Random train/val split across all timesteps")
    
    trained_policy, history = train_act_official_corrected(
        train_dataloader=train_dataloader,
        val_dataloader=val_dataloader,
        checkpoint_dir=checkpoint_dir,
        task=task
    )
    
    print("✅ Training completed!")
    return trained_policy, history, checkpoint_dir

# ========================================
# 🚀 START TRAINING
# ========================================

print("🎯 Starting ACT Training with Official Configuration")
print("📋 Configuration:")
print("   - Chunk size: 100 (official)")
print("   - Hidden dim: 512 (official)")
print("   - Backbone: ResNet34 with FrozenBatchNorm2d")
print("   - Expected checkpoint size: ~350-400 MB")

# Start training
trained_policy, training_history, checkpoint_dir = setup_and_train_act()

print(f"\n🎉 Training Complete!")
print(f"📁 Checkpoints saved to: {checkpoint_dir}")
print(f"📊 Training history available in 'training_history' variable")